In [ ]:
import importlib
from pathlib import Path
import sys

# Mount Google Drive if running in Google Colab
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = "https://github.com/pratik0620/adaptive-federated-aggregation.git"

if IN_COLAB or Path("/content").exists():
    REPO_ROOT = Path("/content/adaptive-federated-healthcare")
    # If alternate clone directory exists and primary does not, use it
    if not REPO_ROOT.exists():
        alt_root = Path("/content/adaptive-federated-aggregation")
        if alt_root.exists() and (alt_root / "src").is_dir():
            REPO_ROOT = alt_root
        else:
            !git clone {REPO_URL} {REPO_ROOT}
    
    # If repository already exists, safely sync with remote and discard local dirty diffs
    if REPO_ROOT.exists() and (REPO_ROOT / ".git").is_dir():
        !git -C {REPO_ROOT} fetch origin main
        !git -C {REPO_ROOT} reset --hard origin/main
else:
    REPO_ROOT = Path.cwd()
    if not (REPO_ROOT / "src").is_dir() and (REPO_ROOT.parent / "src").is_dir():
        REPO_ROOT = REPO_ROOT.parent

# Ensure REPO_ROOT takes top precedence in sys.path
repo_path_str = str(REPO_ROOT)
if repo_path_str in sys.path:
    sys.path.remove(repo_path_str)
sys.path.insert(0, repo_path_str)

# Invalidate import caches and clear cached src modules to prevent stale imports
importlib.invalidate_caches()
for module_name in list(sys.modules):
    if module_name == "src" or module_name.startswith("src."):
        del sys.modules[module_name]

SRC_DIR = REPO_ROOT / "src"
if not SRC_DIR.is_dir():
    raise FileNotFoundError(f"src directory not found at {SRC_DIR}")

print("Repository:", REPO_ROOT)
print("Source modules:", SRC_DIR)

In [ ]:
# Install the packages required by the notebook and shared preprocessing modules.
%pip install -q numpy pandas scikit-learn matplotlib jupyter


# CheXpert preprocessing

This notebook prepares CheXpert as a separate metadata-only federated client. For the configurable target, `1` is positive and `0` is negative; `-1` uncertain and blank/not mentioned are excluded. They are never silently converted to zero.

In [ ]:
import json, platform
import numpy as np
import pandas as pd

from src.config import TARGET_DISEASE, RANDOM_SEED, MAX_POSITIVE_SAMPLES, MAX_NEGATIVE_SAMPLES
from src.dataset_utils import chexpert_clean_binary_labels, standardize_chexpert
from src.preprocessing_utils import (assert_no_leakage, assert_no_study_leakage, build_summary,
                                     require_file, sample_clean_cases, save_splits,
                                     save_summary, set_reproducibility, split_by_patient)
set_reproducibility(RANDOM_SEED)
print('Python:', platform.python_version(), 'pandas:', pd.__version__)

# Dataset-specific Google Drive configuration.
BASE_DRIVE_PATH = Path("/content/drive/MyDrive/Healthcare_FL")
if not BASE_DRIVE_PATH.exists() and (REPO_ROOT / "data").exists():
    BASE_DRIVE_PATH = REPO_ROOT

CHEXPERT_ROOT = BASE_DRIVE_PATH / "data" / "raw" / "CheXpert" if (BASE_DRIVE_PATH / "data" / "raw" / "CheXpert").exists() else BASE_DRIVE_PATH / "raw" / "CheXpert"
TRAIN_CSV = CHEXPERT_ROOT / "train.csv"
IMAGE_ROOT = CHEXPERT_ROOT
OUTPUT_DIR = BASE_DRIVE_PATH / "data" / "processed" / "CheXpert" if (BASE_DRIVE_PATH / "data").exists() else BASE_DRIVE_PATH / "processed" / "CheXpert"

if not TRAIN_CSV.exists():
    raise FileNotFoundError(f"Required CheXpert metadata file not found: {TRAIN_CSV}")

print("Repository:", REPO_ROOT)
print("Google Drive:", BASE_DRIVE_PATH)
print("Dataset:", CHEXPERT_ROOT)
print("Metadata:", TRAIN_CSV)
print("Output Directory:", OUTPUT_DIR)
print("Required CheXpert metadata file exists.")

## Load and inspect
CheXpert contains one label column per pathology. We select the configured target column exactly; release-specific column names should be verified before execution.

In [ ]:
csv_path = require_file(TRAIN_CSV, 'CheXpert train metadata CSV')
# Images are not required for metadata preparation; image_root is only a future path reference.
image_root = IMAGE_ROOT
chexpert = pd.read_csv(csv_path)
target_column = TARGET_DISEASE
if target_column not in chexpert.columns:
    raise KeyError(f'CheXpert target column {target_column!r} is absent. Available columns: {chexpert.columns.tolist()}')
print('studies/images:', len(chexpert))
print('columns:', chexpert.columns.tolist())
display(chexpert.isna().sum().sort_values(ascending=False).head(25).to_frame('missing'))
display(chexpert[target_column].value_counts(dropna=False).sort_index().to_frame('raw_target_distribution'))
clean, counts = chexpert_clean_binary_labels(chexpert, target_column)
print('Counts before filtering:', counts)

In [ ]:
standardized = standardize_chexpert(clean, image_root)
standardized = sample_clean_cases(standardized, MAX_POSITIVE_SAMPLES, MAX_NEGATIVE_SAMPLES, RANDOM_SEED)
print('Clean rows after optional sampling:', len(standardized))
print('unique patients:', standardized['patient_id'].nunique())
display(standardized.head())
splits = split_by_patient(standardized, seed=RANDOM_SEED)
assert_no_leakage(splits)
assert_no_study_leakage(splits)
print('patient and study leakage checks: passed')
save_splits(splits, OUTPUT_DIR)
summary = build_summary(splits, 'CheXpert', {'target_disease': TARGET_DISEASE, 'random_seed': RANDOM_SEED, 'max_positive_samples': MAX_POSITIVE_SAMPLES, 'max_negative_samples': MAX_NEGATIVE_SAMPLES, 'filter_counts_before_filtering': counts})
save_summary(summary, OUTPUT_DIR)
print(json.dumps(summary, indent=2))
display(pd.concat(splits).groupby('target').size().rename('count').to_frame())